# NB08 — Residual SAC + β Ablation

Combines the **base controller** from NB05 with a learned **residual policy**
via SAC.  The final action is: `a = a_base + β × a_residual`.

We train multiple variants with different β values and compare.


## Objective

1. Build a `ResidualActionWrapper` that adds β-scaled learned actions to base controller.
2. Train Residual SAC for β ∈ {0.25, 0.5, 1.0}.
3. Compare variants (ablation table).
4. Select best variant for NB09 evaluation.


## Environment

| Notebook | Goal | Required HW | Min CPU | Min RAM | GPU VRAM | Notes |
|---|---|---|---:|---:|---:|---|
| NB08 | Residual SAC + ablation | GPU | 8 cores | 16 GB | 12-24 GB | Multiple runs |


In [ ]:
import sys, os, platform, json
print(f"Python: {sys.version}"); print(f"OS: {platform.platform()}")
import numpy as np; print(f"NumPy: {np.__version__}")
import torch; print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.cuda.get_device_name(0)}")
import gymnasium as gym; print(f"Gymnasium: {gym.__version__}")
import stable_baselines3 as sb3; print(f"SB3: {sb3.__version__}")
import pandas as pd; print(f"Pandas: {pd.__version__}")


## Imports


In [ ]:
import json, time, copy
import numpy as np
import torch
import gymnasium as gym
import pandas as pd
from pathlib import Path
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.utils import set_random_seed

# Ensure project root is on sys.path so `src.envs` can be imported
PROJECT_ROOT = Path(os.getcwd()).resolve()
if (PROJECT_ROOT / "src").is_dir():
    pass  # already at project root
elif (PROJECT_ROOT.parent / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError(f"Cannot find src/ from {PROJECT_ROOT}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)  # normalize cwd for artifact paths

from src.envs.dishwipe_env import UnitreeG1DishWipeEnv  # registers env


## Config


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IS_GPU = DEVICE == "cuda"

CFG = dict(
    seed=42,
    env_id="UnitreeG1DishWipe-v1",
    control_mode="pd_joint_delta_pos",
    obs_mode="state",
    # ── Training budget (same as NB06/NB07) ──
    total_env_steps=500_000 if IS_GPU else 10_000,
    # ── SAC params ──
    learning_rate=3e-4,
    buffer_size=1_000_000 if IS_GPU else 50_000,
    batch_size=256,
    tau=0.005,
    gamma=0.99,
    ent_coef="auto",
    learning_starts=1000,
    net_arch=[256, 256],
    # ── Residual ──
    beta_values=[0.25, 0.5, 1.0],
    smooth_alpha=0.3,
    # ── Evaluation ──
    eval_episodes=10,
    device=DEVICE,
)
SEED = CFG["seed"]
artifact_dir = Path("artifacts/NB08")
artifact_dir.mkdir(parents=True, exist_ok=True)
print("Config:", json.dumps(CFG, indent=2, default=str))


## Reproducibility


In [ ]:
import random; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
set_random_seed(SEED)
print(f"✅ Seeds set to {SEED}")


## Implementation Steps

### Step 1 — Base controller (from NB05)


In [ ]:
def heuristic_policy(obs, env):
    """Simple proportional controller: push palm toward plate.

    Uses palm position (aligned with env v2 reach reward).
    """
    unwrapped = env.unwrapped if hasattr(env, "unwrapped") else env
    # Try to access env internals — handle wrapped envs
    try:
        palm_pos = unwrapped.agent.robot.links_map["left_palm_link"].pose.p[0].cpu().numpy()
        plate_pos = unwrapped.plate.pose.p[0].cpu().numpy()
    except Exception:
        return np.zeros(env.action_space.shape[-1], dtype=np.float32)

    delta = plate_pos - palm_pos
    act_dim = env.action_space.shape[-1]
    action = np.zeros(act_dim, dtype=np.float32)
    gain = 0.5
    for idx in range(1, min(4, act_dim)):
        action[idx] = np.clip(delta[idx % 3] * gain, -0.3, 0.3)
    return action


class BaseController:
    def __init__(self, env, alpha=0.3):
        self.env = env
        self.alpha = alpha
        self._prev = None
        self.act_dim = env.action_space.shape[-1]

    def reset(self):
        self._prev = np.zeros(self.act_dim, dtype=np.float32)

    def __call__(self, obs):
        raw = heuristic_policy(obs, self.env)
        if self._prev is None:
            self._prev = np.zeros_like(raw)
        smoothed = self.alpha * raw + (1 - self.alpha) * self._prev
        self._prev = smoothed.copy()
        return smoothed

print("✅ BaseController defined.")



### Step 2 — ResidualActionWrapper


In [ ]:
class ResidualActionWrapper(gym.Wrapper):
    """Wraps env so the RL agent outputs a residual action.

    a_final = clip(a_base + beta * a_residual, low, high)

    The wrapper's action space is the same as the original (residual range).
    """

    def __init__(self, env, base_controller, beta=0.5):
        super().__init__(env)
        self.base_controller = base_controller
        self.beta = beta
        self._last_obs = None

    def reset(self, **kwargs):
        self.base_controller.reset()
        obs, info = self.env.reset(**kwargs)
        self._last_obs = obs
        return obs, info

    def step(self, residual_action):
        if isinstance(residual_action, torch.Tensor):
            residual_action = residual_action.cpu().numpy()

        a_base = self.base_controller(self._last_obs)
        a_final = a_base + self.beta * residual_action

        # Clip to action bounds
        low = self.action_space.low
        high = self.action_space.high
        if low is not None and high is not None:
            a_final = np.clip(a_final, low, high)

        obs, reward, terminated, truncated, info = self.env.step(a_final)
        self._last_obs = obs
        return obs, reward, terminated, truncated, info

print("✅ ResidualActionWrapper defined.")


### Step 3 — Train residual SAC for each β


In [ ]:
from mani_skill.utils.wrappers.gymnasium import CPUGymWrapper

ablation_results = []

for beta in CFG["beta_values"]:
    print(f"\n{'='*60}")
    print(f"Training Residual SAC with β={beta}")
    print(f"{'='*60}")

    # Create environment (CPUGymWrapper converts batched obs to standard Gym)
    base_env = gym.make(
        CFG["env_id"], obs_mode=CFG["obs_mode"],
        control_mode=CFG["control_mode"], num_envs=1, render_mode=None,
    )
    base_env = CPUGymWrapper(base_env)
    base_ctrl = BaseController(base_env, alpha=CFG["smooth_alpha"])
    train_env = ResidualActionWrapper(base_env, base_ctrl, beta=beta)

    # Create SAC
    set_random_seed(SEED)
    model = SAC(
        "MlpPolicy", train_env,
        learning_rate=CFG["learning_rate"],
        buffer_size=CFG["buffer_size"],
        batch_size=CFG["batch_size"],
        tau=CFG["tau"], gamma=CFG["gamma"],
        ent_coef=CFG["ent_coef"],
        learning_starts=CFG["learning_starts"],
        policy_kwargs=dict(net_arch=CFG["net_arch"]),
        verbose=0, seed=SEED, device=DEVICE,
    )

    t0 = time.time()
    model.learn(total_timesteps=CFG["total_env_steps"], progress_bar=True)
    train_time = time.time() - t0
    print(f"  Trained in {train_time:.1f}s")

    # Save model
    model_name = f"residual_sac_beta{beta}"
    model.save(str(artifact_dir / model_name))

    # Evaluate
    eval_env = gym.make(
        CFG["env_id"], obs_mode=CFG["obs_mode"],
        control_mode=CFG["control_mode"], num_envs=1, render_mode=None,
    )
    eval_env = CPUGymWrapper(eval_env)
    eval_ctrl = BaseController(eval_env, alpha=CFG["smooth_alpha"])
    eval_wrapped = ResidualActionWrapper(eval_env, eval_ctrl, beta=beta)

    rewards, cleaned_list = [], []
    for ep in range(CFG["eval_episodes"]):
        obs, info = eval_wrapped.reset(seed=SEED + 3000 + ep)
        total_rew = 0.0
        for step in range(1000):
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = eval_wrapped.step(action)
            total_rew += reward.item() if hasattr(reward, "item") else float(reward)
            if (terminated.any() if hasattr(terminated, "any") else terminated):
                break
        ratio = info.get("cleaned_ratio", torch.tensor([0.0]))
        rewards.append(total_rew)
        cleaned_list.append(ratio.item() if hasattr(ratio, "item") else float(ratio))

    result = dict(
        beta=beta,
        mean_reward=float(np.mean(rewards)),
        std_reward=float(np.std(rewards)),
        mean_cleaned=float(np.mean(cleaned_list)),
        std_cleaned=float(np.std(cleaned_list)),
        train_time=train_time,
    )
    ablation_results.append(result)
    print(f"  β={beta}: reward={result['mean_reward']:.3f}, "
          f"cleaned={result['mean_cleaned']:.4f}")

    eval_env.close()
    base_env.close()

print("\n✅ All β variants trained.")


### Step 4 — Ablation table


In [ ]:
df = pd.DataFrame(ablation_results)
print(df.to_string(index=False))

# Save
ablation_path = artifact_dir / "ablation_beta_table.csv"
df.to_csv(ablation_path, index=False)
print(f"\n✅ Saved: {ablation_path}")

# Select best
best = df.loc[df["mean_reward"].idxmax()]
print(f"\n🏆 Best variant: β={best['beta']}, reward={best['mean_reward']:.3f}")


### Step 5 — Visualise ablation


In [ ]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

betas = [r["beta"] for r in ablation_results]
means = [r["mean_reward"] for r in ablation_results]
stds = [r["std_reward"] for r in ablation_results]
ax1.bar(range(len(betas)), means, yerr=stds, tick_label=[f"β={b}" for b in betas],
        color=["#1976D2", "#388E3C", "#F57C00"], capsize=5)
ax1.set_ylabel("Mean Eval Reward")
ax1.set_title("Reward by β")

cleaned_means = [r["mean_cleaned"] for r in ablation_results]
cleaned_stds = [r["std_cleaned"] for r in ablation_results]
ax2.bar(range(len(betas)), cleaned_means, yerr=cleaned_stds,
        tick_label=[f"β={b}" for b in betas],
        color=["#1976D2", "#388E3C", "#F57C00"], capsize=5)
ax2.set_ylabel("Mean Cleaned Ratio")
ax2.set_title("Cleaning by β")

fig.suptitle("NB08 — Residual SAC β Ablation")
fig.tight_layout()
fig.savefig(str(artifact_dir / "ablation_plot.png"), dpi=120, bbox_inches="tight")
plt.close("all")
print("✅ Saved: ablation_plot.png")


### Step 6 — MLflow


In [ ]:
try:
    import mlflow; from dotenv import load_dotenv; load_dotenv(".env.local")
    uri = os.environ.get("MLFLOW_TRACKING_URI", "")
    if uri:
        mlflow.set_tracking_uri(uri)
        mlflow.set_experiment("dishwipe_unitree_g1")
        with mlflow.start_run(run_name="NB08_ResidualSAC_ablation"):
            for r in ablation_results:
                b = r["beta"]
                mlflow.log_metric(f"beta{b}_reward", r["mean_reward"])
                mlflow.log_metric(f"beta{b}_cleaned", r["mean_cleaned"])
            mlflow.log_metric("best_beta", float(best["beta"]))
            mlflow.log_artifacts(str(artifact_dir), artifact_path="NB08")
        print("✅ MLflow run logged.")
except Exception as e:
    print(f"⚠️ MLflow: {e}")


## Results

- Residual SAC trained for β ∈ {0.25, 0.5, 1.0}
- Ablation table shows how β affects reward and cleaning performance
- Best variant selected for NB09 final evaluation


## Artifacts

| File | Description |
|------|-------------|
| `artifacts/NB08/residual_sac_beta*.zip` | Models for each β |
| `artifacts/NB08/ablation_beta_table.csv` | Ablation comparison table |
| `artifacts/NB08/ablation_plot.png` | Bar chart comparison |


## Cleanup


In [ ]:
print('✅ NB08 complete.')


## References

- Silver et al. (2018) — Residual Policy Learning
- Johannink et al. (2019) — Residual RL for real robots
- Haarnoja et al. (2018) — SAC
